# 01 · Features, directions and superposition

**Paper:** Introduction and "Definitions and Motivation: Features, Directions, and
Superposition", pp. 1–8.

**Learning goals**

1. Say what it means for a representation to be *linear*: features are directions (pp. 5–6).
2. Tell a privileged basis from a non-privileged one, and show it with a rotation (pp. 6–7).
3. State the superposition hypothesis and its formal test, WᵀW not invertible (pp. 7–8).
4. See why high-dimensional spaces have room for many *almost*-orthogonal directions, and what
   interference costs (p. 7).

Nothing is trained here: we build weight matrices by hand so every number has an explanation.

In [ ]:
# Runs locally and in Colab. In a fresh environment (e.g. Colab), install the package first.
# Replace <OWNER> with the GitHub owner of your copy of the repository.
try:
    import superposition  # noqa: F401
except ImportError:
    %pip install -q "git+https://github.com/<OWNER>/superposition.git"

In [ ]:
import math

import torch

from superposition import metrics
from superposition.models import TiedModel

torch.set_printoptions(precision=3, sci_mode=False)

## 1. Features as directions (pp. 5–6)

In a **linear representation**, feature i has a direction Wᵢ, and features x₁, x₂, … are
represented by the vector h = Σᵢ xᵢ Wᵢ = Wx. Reading a feature back is a dot product with its
direction. With orthonormal directions, Wᵀh recovers x exactly.

**Predict:** if we rotate every direction by the same orthogonal matrix Q (W → QW), does the
reconstruction WᵀWx change?

In [ ]:
g = torch.Generator().manual_seed(0)
W = torch.eye(3)  # three features, three dimensions, basis-aligned
Q, _ = torch.linalg.qr(torch.randn(3, 3, generator=g))
x = torch.tensor([0.7, 0.0, 0.2])

print("reconstruction with W :", W.T @ W @ x)
print("reconstruction with QW:", (Q @ W).T @ (Q @ W) @ x)
print("max |WᵀW − (QW)ᵀ(QW)|:", ((W.T @ W) - (Q @ W).T @ (Q @ W)).abs().max().item())

Nothing changes: (QW)ᵀ(QW) = WᵀQᵀQW = WᵀW. The individual hidden dimensions carry no special
meaning; only the directions relative to each other do.

## 2. Privileged vs non-privileged bases (pp. 6–7, 31)

A basis is **privileged** when something in the architecture treats its directions
differently, typically an elementwise nonlinearity. Then (and only then) it makes sense to ask
whether a *neuron* is interpretable.

**Predict:** the ReLU output model applies its ReLU to the *output* x′, the ReLU hidden model
also applies one to h. Which of the two gives identical outputs after rotating the hidden space?

In [ ]:
def outputs(kind: str, W: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    model = TiedModel.from_kind(kind, n_features=W.shape[1], n_hidden=W.shape[0], n_inst=1)
    with torch.no_grad():
        model.W.copy_(W[None])
    return model(x[None, None])[0, 0].detach()


W = torch.randn(3, 5, generator=g) * 0.6  # 5 features in 3 dimensions
x = torch.rand(5, generator=g)
for kind in ("relu_output", "relu_hidden"):
    diff = (outputs(kind, W, x) - outputs(kind, Q @ W, x)).abs().max().item()
    print(f"{kind:<12} max output change after rotation: {diff:.4f}")

The ReLU output model does not care (its hidden space is not privileged); the ReLU hidden
model does, because ReLU(QWx) ≠ Q·ReLU(Wx). That is why the paper studies W *directly* only for
the ReLU hidden model (p. 31) and studies WᵀW for the others (p. 10).

## 3. Superposition (pp. 7–8)

**Superposition** means representing more features than dimensions. Formally (p. 8), a linear
representation exhibits superposition when WᵀW is not invertible. Take five unit directions in
two dimensions, a regular pentagon, the geometry E00 finds.

**Predict:** what is the rank of the 5×5 matrix WᵀW? What does `is_superposition` say?

In [ ]:
angles = 2 * math.pi * torch.arange(5) / 5
pentagon = torch.stack([angles.cos(), angles.sin()])  # shape (hidden=2, feat=5)

gram = metrics.gram(pentagon[None])[0]
print("WᵀW =\n", gram)
print("rank:", torch.linalg.matrix_rank(gram).item())
print("superposition?", metrics.is_superposition(pentagon[None]).item())
print("Dᵢ (feature dimensionality):", metrics.feature_dimensionality(pentagon[None])[0])

Rank 2, so WᵀW is not invertible: superposition. Each feature gets 2/5 of a dimension
(Dᵢ = 2/5, p. 19); notebook 05 returns to these fractions.

### Interference, and how a ReLU and a negative bias filter it (pp. 7, 13)

Activate feature 0 alone. The linear readout WᵀWx gives feature 0 its value, but also leaks
cos 72° ≈ 0.31 into its two neighbours and cos 144° ≈ −0.81 into the other two: that leak is
**interference**.

**Predict:** after ReLU(WᵀWx + b) with b = −0.3 for every feature, which of the leaks survive?

In [ ]:
x = torch.tensor([1.0, 0.0, 0.0, 0.0, 0.0])
readout = gram @ x
print("linear readout WᵀWx      :", readout)
print("ReLU(WᵀWx + b), b = −0.3 :", torch.relu(readout - 0.3))

The negative interference is removed by the ReLU for free, and the bias shaves the positive
interference down to almost nothing. Feature 0 itself also loses 0.3, which is why trained
pentagons have norms a little above 1 (p. 23; notebook 02, exercise 3, works it out).

This only works if features are **sparse**: when two features are active together, their
interference adds up and the filter no longer copes (p. 7, right-hand figure).

**Predict:** activate features 1 and 4 together, the two neighbours of feature 0. Feature 0
is off (x₀ = 0): what does the model output for it?

In [ ]:
x = torch.tensor([0.0, 1.0, 0.0, 0.0, 1.0])
print("ReLU(WᵀWx − 0.3):", torch.relu(gram @ x - 0.3))

Each neighbour leaks 0.31 into feature 0; together they leak 0.62, and the bias only removes
0.3, so the model reports feature 0 at about 0.32 although it is absent. Worse, features 1 and
4 sit 144° apart, so each leaks −0.81 into the other and the ReLU erases *both* of the features
that are actually present. Interference *compounds* when several features are active at once.
The sparser the features, the rarer this is, which is why sparsity decides whether
superposition pays off.

## 4. Almost-orthogonal vectors (p. 7)

In m dimensions only m vectors can be exactly orthogonal, but *exponentially many* can be
almost orthogonal (Johnson–Lindenstrauss). Two random unit vectors in m dimensions have a
root-mean-square cosine of exactly 1/√m.

**Predict:** for 4m random directions in m dimensions, how do the typical and the largest
|cos| between two of them change as m goes from 4 to 400?

In [ ]:
for m in (4, 16, 64, 400):
    n = 4 * m
    W = torch.randn(1, m, n, generator=g)
    cos = metrics.cosine_similarity_matrix(W)[0]
    off = cos[~torch.eye(n, dtype=torch.bool)].abs()
    rms = off.pow(2).mean().sqrt()
    print(
        f"m = {m:>3}, n = {n:>4}: rms cos = {rms:.3f}   1/√m = {1 / math.sqrt(m):.3f}   max |cos| = {off.max():.3f}"
    )

With enough dimensions, many more features than dimensions can coexist with small pairwise
interference. The paper's hypothesis is that networks exploit exactly this when features are
sparse enough that the interference rarely matters.

## 5. A hierarchy of properties (p. 8)

| Property | Meaning | Test here |
|---|---|---|
| Decomposability | activations decompose into independently understandable features | (assumed by construction) |
| Linearity | features are directions | `h = Wx` |
| Superposition vs not | WᵀW not invertible vs invertible | `metrics.is_superposition` |
| Basis-aligned | every Wᵢ is one-hot (needs a privileged basis) | one nonzero per column |

## Exercises

**1.** Build six features in two dimensions as a regular hexagon. What is Dᵢ, and does it
match the formula for a unit-norm tight frame (Dᵢ = m/n)?

<details><summary>Solution</summary>

```python
angles = 2 * math.pi * torch.arange(6) / 6
hexagon = torch.stack([angles.cos(), angles.sin()])
print(metrics.feature_dimensionality(hexagon[None]))  # all 1/3 = m/n
```

A regular hexagon is a tight frame, so Dᵢ = 2/6 = 1/3. (Opposite vertices are antipodal, but
each feature still shares its line with two neighbours at 60°.)
</details>

**2.** Two features stored as an antipodal pair, W = [[1, −1]] (m = 1). Compute the
superposition score and Dᵢ. Why is an antipodal pair a cheap form of superposition under a
ReLU?

<details><summary>Solution</summary>

Score 1 and Dᵢ = 1/2 for both. When one feature is active, the other's readout is negative, and
ReLU(negative) = 0: the interference costs nothing. It only hurts when both are active at once,
which is rare when features are sparse.
</details>

**3.** Take the pentagon above and make feature 4's direction have norm 0.1. Does the formal
test still report superposition? What about with the default threshold `threshold=0.5`?

<details><summary>Solution</summary>

`is_superposition` only considers *represented* features (‖Wᵢ‖ > 0.5). Four unit vectors in two
dimensions are still linearly dependent, so the answer stays True; with only two represented
features at 72° it would be False, even though they are not orthogonal.
</details>

**4.** Using `outputs(...)` above, check that the *linear* model is also rotation-invariant.
Then find a transformation of W that is *not* orthogonal and changes the linear model's
output.

<details><summary>Solution</summary>

For `kind="linear"` the difference after rotation is 0. Scaling the hidden space, W → 2W,
changes WᵀW → 4WᵀW and hence the output: only orthogonal maps are symmetries of these models.
</details>